In [ ]:
#@title  Connect { display-mode: "form" }
import builtins, subprocess, sys
from pathlib import Path
from IPython.display import display
import ipywidgets as W

_S = globals().setdefault("_ANVIL", {})
_repo = Path("/content/anvil")
_status = W.HTML()
_bar = W.IntProgress(value=0, min=0, max=5, bar_style="info", layout=W.Layout(width="360px"))
_box = W.VBox([
    W.HTML("<div style='font:600 15px system-ui;color:#111'>Setting up your workspace</div>"
           "<div style='font:13px system-ui;color:#666;margin-top:2px'>Takes a moment. You only do this once per session.</div>"),
    _bar,
    _status,
])
display(_box)

def _tick(n, msg):
    _bar.value = n
    _status.value = f"<div style='font:13px system-ui;color:#444;margin-top:4px'>{msg}</div>"

def _done(msg, ok=True):
    _bar.value = _bar.max
    _bar.bar_style = "success" if ok else "danger"
    color = "#0a7d2c" if ok else "#b00020"
    _status.value = f"<div style='font:600 14px system-ui;color:{color};margin-top:6px'>{msg}</div>"

try:
    _tick(1, "Preparing Anvil…")
    if not (_repo / "src" / "anvil").exists():
        _tick(2, "Downloading the latest Anvil files…")
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/motionssalt/anvil.git", str(_repo)], check=True)
    sys.path.insert(0, str(_repo / "src"))
    _tick(3, "Installing packages…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_repo / "requirements.txt")], check=True)
    _tick(4, "Checking GPU and loading the model…")
    from anvil.setup import setup_runtime
    builtins.ANVIL = setup_runtime(install=False)
    _S["ready"] = True
    _done("Ready. Continue to the next cell.")
except Exception as _e:
    _done(f"Setup failed: {_e}", ok=False)
    raise


In [ ]:
#@title  Upload a file { display-mode: "form" }
import builtins
from pathlib import Path
from IPython.display import display
import ipywidgets as W

_work = Path("/content/anvil_workspace")
_input = _work / "input"
_input.mkdir(parents=True, exist_ok=True)
_uploader = W.FileUpload(accept="*/*", multiple=True, description="Choose files", button_style="primary", layout=W.Layout(width="180px"))
_info = W.HTML("<div style='font:13px system-ui;color:#888;margin-top:8px'>No files selected yet.</div>")
_card = W.VBox([
    W.HTML("<div style='font:600 15px system-ui;color:#111'>Give Anvil something to work with</div>"
           "<div style='font:13px system-ui;color:#666;margin-top:2px'>Upload images, documents, data, audio, or video.</div>"),
    W.HBox([_uploader]),
    _info,
], layout=W.Layout(border="1px solid #e4e4e7", border_radius="10px", padding="16px 18px", width="520px"))
display(_card)

def _on_upload(change):
    selected = _uploader.value
    if not selected:
        return
    records = selected.values() if isinstance(selected, dict) else selected
    paths = []
    for record in records:
        name = record.get("name", "upload.bin") if isinstance(record, dict) else record.name
        content = record.get("content", b"") if isinstance(record, dict) else record.content
        dest = _input / Path(name).name
        dest.write_bytes(bytes(content))
        paths.append(str(dest))
    builtins.ANVIL_FILES = paths
    names = ", ".join(Path(p).name for p in paths)
    _info.value = f"<div style='font:600 14px system-ui;color:#0a7d2c;margin-top:8px'>{names} · ready for Anvil</div>"

_uploader.observe(_on_upload, names="value")


In [ ]:
#@title  Chat with Anvil { display-mode: "form" }
from anvil.agent import launch
launch()


In [ ]:
#@title  Download { display-mode: "form" }
import builtins
from pathlib import Path
from IPython.display import display, HTML

_files = [Path(p) for p in getattr(builtins, "ANVIL_FILES", []) if Path(p).exists()]
if _files:
    display(HTML("<div style='font:600 15px system-ui;color:#111'>Files are ready</div>"))
    display(HTML("<div style='font:13px system-ui;color:#666;margin-top:4px'>Use the Downloads panel in the Anvil chat window to retrieve generated files.</div>"))
else:
    display(HTML("<div style='font:13px system-ui;color:#666'>Generated files appear in the Downloads panel inside Anvil.</div>"))
